### `Q&A` PDF using open-source LLM and Huggingface embeddings`


RAG implementation

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.vectorstores import FAISS, Chroma
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
import os
from dotenv import load_dotenv
from langchain.embeddings import HuggingFaceEmbeddings
load_dotenv()
import warnings
warnings.filterwarnings("ignore")

/workspaces/GenAI-Krishnaik/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

llm = ChatGroq(groq_api_key=groq_api_key,model_name='llama-3.1-8b-instant')


prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant. Answer the question using only the provided context.
extract the key points in answer. Do not make up information.

Context:
{context}

Question:
{question}

Answer:
"""
)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") #OllamaEmbeddings()

loader = PyPDFLoader("attention.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
len(chunks)

vectorstore = Chroma.from_documents(chunks, embeddings)
print("Vector DB is Ready")

# doc_chain = create_stuff_documents_chain(llm, prompt)
# retriever = vectorstore.as_retriever()
# retriver_chain = create_retrieval_chain(retriever,doc_chain)

#retriver_chain.invoke({"input":"You are required to answer", "question":"what is attention all you need?"})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm, 
    retriever=vectorstore.as_retriever(),
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=False
    )


result = qa_chain.invoke("What are the key points discuused in this")
    

Vector DB is Ready


In [15]:
qa_chain.invoke("What are the key points discuused in this paper?")

{'query': 'What are the key points discuused in this paper?',
 'result': "Key points not explicitly mentioned but the title of the paper 'Attention Is All You Need' suggests the following key points are discussed:\n\n1. Attention mechanism in the paper: This is the primary focus of the paper suggested by the title.\n2. The need for attention in AI models: This can be inferred as the paper's title implies that attention is necessary.\n3. A new approach to AI models that uses attention: The paper likely discusses a new method or technique that incorporates attention in AI models.\n\nThe provided text does not explicitly mention the key points discussed in the paper. The repeated permission statement suggests that the main content of the paper is not provided."}

In [14]:
result

{'query': 'what is this paper discussing?',
 'result': 'This paper is discussing the concept of "Attention Is All You Need" which is the title of the paper. However, there is no explicit information provided in the given context to describe the main discussion of the paper.\n\nKey points:\n- The paper title is "Attention Is All You Need".\n- The authors are Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, and Aidan N. Gomez.\n- The paper grants permission to reproduce its tables and figures for journalistic or scholarly works.'}

## `LECL` Chain

In [21]:
from langchain.schema.output_parser import StrOutputParser

lecl_prompt = ChatPromptTemplate.from_template(""" Summarize the doc on the {topic}. response 'No result', if not found""")
lecl_llm = llm
parser = StrOutputParser()

chain = lecl_prompt | lecl_llm | parser

In [23]:
chain.invoke({"topic":"Random Forest"})

"Random Forest is an ensemble learning method that combines multiple decision trees to improve the accuracy and robustness of predictions. Here's a summary of the key points:\n\n**Key Components:**\n\n1. **Decision Trees:** Random Forest consists of multiple decision trees, each of which is trained on a random subset of the data.\n2. **Bootstrap Sampling:** Each decision tree is trained on a random subset of the data, drawn with replacement (bootstrap sampling).\n3. **Random Feature Selection:** At each node, a random subset of features is selected for splitting, rather than using all features.\n4. **Voting:** The final prediction is made by aggregating the predictions of all decision trees, using a voting system (e.g., majority vote).\n\n**Benefits:**\n\n1. **Improved Accuracy:** Random Forests can achieve higher accuracy than individual decision trees, especially when the data is complex or noisy.\n2. **Robustness:** Random Forests are more robust to overfitting and outliers, as the 